In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)

In [9]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)

from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [10]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes. You can still join. If you want a certificate, make sure to submit your project while submissions are still open. You can start learning and submit homework while the form is open—no need to wait for any confirmation email.'

In [11]:
assistant.total_cost()

0.001464

In [12]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [13]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes. You can still join. If you want a certificate, make sure to submit your project while submissions are still open. You can start learning and submit homework while the form is open—no need to wait for any confirmation email.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [15]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes. You can join now. If you want a certificate, make sure to submit your project while submissions are still open. You don’t need a confirmation email—just start learning and submit homework while the form is open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [16]:
assistant.reset_usage()

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/395 [00:00<?, ?it/s]

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
assistant.total_cost()

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("rag-answers-new.csv", index=False)